# Task map

Every policy's success on each of the 40 LIBERO tasks, tasks sorted from easiest to hardest (mean over policies): if the policies fail on the same tasks, the rows look alike.

In [ ]:
import json
import re
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt

import utils as u

FS_SMALL, FS_TINY = 9, 7  # pt: \small, \scriptsize of the paper
PAGE_WIDTH = 5.5  # in: \textwidth of the paper (\linewidth of a figure)
ROOT = Path.cwd().parent
data_path = ROOT / "results"
figs_path = ROOT / "analysis/figs"
style_path = ROOT / "analysis/figure.mplstyle"

In [ ]:
data = json.loads((data_path / "data.json").read_text())
figs_path.mkdir(exist_ok=True)
plt.style.use(style_path)

In [ ]:
panels = [(bench, *u.task_matrix(data, bench)) for bench in ("libero", "libero_plus")]
n = len(u.POLICIES)

In [ ]:
fig, axes = plt.subplots(len(panels), 1, figsize=(PAGE_WIDTH, 0.432 + 0.1296 * n * len(panels) + 0.288 * len(panels)),
                         layout="constrained", squeeze=False)  # fmt: skip
axes = axes[:, 0]
for ax, (bench, rows, m) in zip(axes, panels, strict=True):
    ax.imshow(m, cmap="Blues", vmin=0, vmax=1, aspect="auto", interpolation="nearest")
    ax.set_yticks(range(len(rows)), [re.sub(r"π([\d.]+)", r"$\\pi_{\1}$", lab) for lab in rows], fontsize=FS_TINY)
    for tick, lab in zip(ax.get_yticklabels(), rows, strict=True):
        if lab.startswith("k-NN"):
            tick.set_color(u.Colour.KNN)
            tick.set_fontweight("bold")
    ax.set_xticks([])
    for sp in ax.spines.values():
        sp.set_visible(False)
    ax.tick_params(length=0)
    ax.set_title("LIBERO" if bench == "libero" else "LIBERO-Plus", fontsize=FS_SMALL,
                 loc="left")  # fmt: skip
axes[-1].set_xlabel("Tasks, sorted by mean success over policies", fontsize=FS_SMALL)
sm = mpl.cm.ScalarMappable(cmap="Blues", norm=mpl.colors.Normalize(0, 100))
cb = fig.colorbar(sm, ax=axes, location="right", shrink=0.6, pad=0.01, aspect=25)
cb.set_label("Task success (%)", fontsize=FS_TINY)
cb.ax.tick_params(labelsize=FS_TINY)
cb.outline.set_visible(False)
fig.savefig(figs_path / "task_map.pdf")